# Data exploration, canonical split, and popularity baseline

This notebook establishes the data and evaluation contract used throughout the project. It validates the MovieLens input, partitions each user's rating history chronologically, and measures a popularity baseline. Run it before notebooks 02–07: downstream notebooks consume the split files produced here rather than reconstructing their own date cutoffs.

The prediction question is: **given an existing user's earlier ratings, which movies that they later rated highly can the recommender retrieve?** The split preserves earlier-to-later order within each user's history and keeps the same users represented in all three portions.

| Portion | Assignment within each user's ordered history | Use |
|---|---|---|
| Training | First 60%, rounded down | Fit similarities, representations, and popularity counts |
| Validation | From the 60% boundary to the 80% boundary | Compare models and choose settings |
| Test | Remaining ratings | Final assessment after validation decisions are fixed |

The protocol is an existing-user offline benchmark. Users' calendar timelines overlap: another user's training interaction may occur after a particular user's held-out interaction. It therefore does not reproduce deployment at a single historical date. Rating timestamps record submission time, which may differ from when a movie was watched. Identical timestamps are ordered deterministically by `movieId`; that tie-break does not imply a known viewing order.


## Load the source tables

Resolve the project from its data directory so the notebook can run from the project root or the `notebooks` directory. The source tables remain unchanged; generated artifacts use a split-specific directory to keep results from older evaluation protocols separate.


In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

SPLIT_ID = "per_user_chronological_60_20_20_v1"
POSITIVE_THRESHOLD = 4.0
TOP_K = 10

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "raw" / "ml-latest-small").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Run this notebook inside the project containing "
        "data/raw/ml-latest-small/movies.csv and ratings.csv."
    )

raw_dir = data_dir = project_root / "data" / "raw" / "ml-latest-small"
processed_dir = project_root / "data" / "processed" / SPLIT_ID
processed_dir.mkdir(parents=True, exist_ok=True)

movies = pd.read_csv(raw_dir / "movies.csv")
ratings = pd.read_csv(raw_dir / "ratings.csv")

print("Python:", sys.executable)
print("Project root:", project_root)
display(movies.head())
display(ratings.head())


## Check keys, missing values, and coverage

A rating is identified by `(userId, movieId)` in this dataset. Duplicate pairs would make event overlap and recall denominators ambiguous, so the pipeline rejects them before splitting. The catalog join must be many-to-one and every rated movie must have catalog metadata. Unrated catalog movies are retained for exploration but are outside the training candidate set used by these models.


In [ ]:
assert not movies["movieId"].duplicated().any(), "Duplicate movie IDs."
assert not ratings.duplicated(["userId", "movieId"]).any(), (
    "Repeated user–movie pairs require an explicit handling policy."
)
assert not ratings[["userId", "movieId", "rating", "timestamp"]].isna().any().any()
assert not movies[["movieId", "title", "genres"]].isna().any().any()
assert ratings["movieId"].isin(movies["movieId"]).all()
assert ratings["rating"].between(0.5, 5.0).all()

print("Movies:", len(movies))
print("Ratings:", len(ratings))
print("Users:", ratings["userId"].nunique())
print("Rated movies:", ratings["movieId"].nunique())

movies_without_ratings = movies.loc[
    ~movies["movieId"].isin(ratings["movieId"])
].copy()
print("Catalog movies without ratings:", len(movies_without_ratings))
display(movies_without_ratings.head())
display(ratings["rating"].value_counts().sort_index().rename("rating_count"))


## Attach metadata and interpret timestamps

The validated join adds titles and genres without multiplying interactions. Convert Unix seconds to timezone-aware UTC values for diagnostics and chronological checks; splitting still sorts the original integer timestamps.


In [ ]:
ratings_with_movies = ratings.merge(
    movies,
    on="movieId",
    how="left",
    validate="many_to_one",
)
assert len(ratings_with_movies) == len(ratings)

ratings_with_movies["rated_at"] = pd.to_datetime(
    ratings_with_movies["timestamp"], unit="s", utc=True
)
display(ratings_with_movies.head())
print("Distinct timestamps:", ratings_with_movies["timestamp"].nunique())
print("Repeated timestamps:", ratings_with_movies["timestamp"].duplicated().sum())


## Describe rating activity over time

Annual counts show the source dataset's calendar coverage. These are descriptive diagnostics: the split uses positions within each user's history, so annual activity does not determine a shared training or validation end date. The first and last years may be partial years.


In [ ]:
import matplotlib.pyplot as plt

ratings_by_year = (
    ratings_with_movies.resample("YS", on="rated_at")
    .agg(rating_count=("rating", "size"))
)
ratings_by_year.index = ratings_by_year.index.year
ratings_by_year.index.name = "year"
ratings_by_year["percent_of_ratings"] = (
    100 * ratings_by_year["rating_count"] / len(ratings_with_movies)
)
display(ratings_by_year.round(2))

ax = ratings_by_year["rating_count"].plot(kind="bar", figsize=(11, 4))
ax.set_title("Rating activity by submission year")
ax.set_xlabel("Year")
ax.set_ylabel("Number of ratings")
plt.tight_layout()
plt.show()


## Assign the per-user chronological split

Order all ratings by user, submission timestamp, and movie ID. `cumcount` supplies each rating's zero-based position, and `transform("size")` repeats its user's history length. Whole-number boundaries assign every rating exactly once; rounding means aggregate row proportions will be close to 60/20/20 rather than exact.

Split **all** ratings before applying the positive-feedback threshold. Low ratings are still part of observed history and must be excluded from later recommendations. The threshold determines positive examples during modeling and evaluation, not membership of the split.


In [ ]:
ordered_ratings = ratings_with_movies.sort_values(
    ["userId", "timestamp", "movieId"]
).copy()
position = ordered_ratings.groupby("userId").cumcount()
user_total = ordered_ratings.groupby("userId")["movieId"].transform("size")

train_end = (user_total * 0.60).astype(int)
validation_end = (user_total * 0.80).astype(int)

train_ratings = ordered_ratings.loc[position < train_end].copy()
validation_ratings = ordered_ratings.loc[
    (position >= train_end) & (position < validation_end)
].copy()
test_ratings = ordered_ratings.loc[position >= validation_end].copy()

split_frames = {
    "train": train_ratings,
    "validation": validation_ratings,
    "test": test_ratings,
}
split_summary = pd.DataFrame({
    name: {"ratings": len(frame), "users": frame["userId"].nunique()}
    for name, frame in split_frames.items()
}).T
split_summary.index.name = "split"
display(split_summary)


## Verify event separation, coverage, and chronology

These assertions protect downstream evaluation from duplicate or missing interactions. Each user's latest training timestamp must be no later than their earliest validation timestamp, and the same ordering applies between validation and test. Equal timestamps are valid under the documented tie-break.

The existing-user benchmark requires every source user to have observations in all portions. This condition holds for the supplied MovieLens dataset; a future dataset containing very short histories would need an explicit eligibility policy rather than silently changing this benchmark.


In [ ]:
assert ratings_with_movies.index.is_unique
combined = pd.concat(list(split_frames.values()))
assert combined.index.is_unique, "Some rating rows occur in multiple splits."
assert set(combined.index) == set(ratings_with_movies.index)
assert not combined.duplicated(["userId", "movieId"]).any()

source_users = set(ratings["userId"])
for name, frame in split_frames.items():
    assert set(frame["userId"]) == source_users, (
        f"{name}: at least one user has no observations."
    )

train_latest = train_ratings.groupby("userId")["timestamp"].max()
validation_earliest = validation_ratings.groupby("userId")["timestamp"].min()
validation_latest = validation_ratings.groupby("userId")["timestamp"].max()
test_earliest = test_ratings.groupby("userId")["timestamp"].min()
assert (train_latest <= validation_earliest).all()
assert (validation_latest <= test_earliest).all()
print("All split integrity checks passed.")


## Persist the split contract

Write the canonical interaction tables and a manifest describing the split, source fingerprints, and output fingerprints. The saved CSVs contain interaction columns only; models join catalog metadata separately. Downstream notebooks verify these fingerprints and load the saved training and validation rows, which prevents a notebook from silently reverting to the former global date-cutoff protocol.


In [ ]:
# Write all partitions once; downstream notebooks consume these exact files.
import hashlib
import json

SPLIT_ID = "per_user_chronological_60_20_20_v1"
POSITIVE_THRESHOLD = 4.0
TOP_K = 10
raw_dir = data_dir
processed_dir = project_root / "data" / "processed" / SPLIT_ID
processed_dir.mkdir(parents=True, exist_ok=True)

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

split_files = {}
for name, frame in (
    ("train", train_ratings),
    ("validation", validation_ratings),
    ("test", test_ratings),
):
    path = processed_dir / f"{name}_ratings.csv"
    frame[["userId", "movieId", "rating", "timestamp", "rated_at"]].to_csv(
        path, index=False
    )
    split_files[path.name] = {
        "sha256": file_sha256(path),
        "rows": len(frame),
        "users": int(frame["userId"].nunique()),
    }

raw_sha256 = {
    name: file_sha256(raw_dir / name)
    for name in ("movies.csv", "ratings.csv")
}
split_fingerprint = hashlib.sha256(
    json.dumps({
        "split_id": SPLIT_ID,
        "raw_sha256": raw_sha256,
        "files": split_files,
    }, sort_keys=True).encode("utf-8")
).hexdigest()
split_manifest = {
    "split_id": SPLIT_ID,
    "split_fingerprint": split_fingerprint,
    "strategy": "chronological within each user",
    "sort_columns": ["userId", "timestamp", "movieId"],
    "train_fraction": 0.60,
    "validation_boundary": 0.80,
    "rounding": "floor each cumulative boundary",
    "positive_threshold": POSITIVE_THRESHOLD,
    "top_k": TOP_K,
    "candidate_policy": "movies observed in the fitting interactions",
    "recall_denominator": "all held-out positives, including unavailable movies",
    "files": split_files,
    "raw_sha256": raw_sha256,
}
(processed_dir / "split_manifest.json").write_text(
    json.dumps(split_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

def save_validation_results(results, filename):
    """Attach the split and evaluation contract to each per-user result row."""
    saved = results.copy()
    saved["split_id"] = SPLIT_ID
    saved["split_fingerprint"] = split_manifest["split_fingerprint"]
    saved["evaluation_split"] = "validation"
    saved["positive_threshold"] = POSITIVE_THRESHOLD
    saved["k"] = TOP_K
    saved.to_csv(processed_dir / filename, index=False)

print("Canonical split saved:", processed_dir)
print("Split fingerprint:", split_fingerprint)


## Inspect validation eligibility and candidate availability

A user is eligible for Recall@10 only when they have at least one validation rating of 4.0 or higher. All eligible users receive equal weight in the final mean. Every held-out positive remains in the denominator, including movies absent from training; this exposes the cost of restricting recommendations to the training catalog.

Training history and **liked** training history are different: the former is guaranteed by the split, while the latter may be absent and requires a popularity fallback in personalized models. These checks inspect validation only; test preferences remain reserved for the final assessment.


In [ ]:
validation_positives = validation_ratings.loc[
    validation_ratings["rating"] >= POSITIVE_THRESHOLD
].copy()
evaluation_user_ids = sorted(validation_positives["userId"].unique())
assert evaluation_user_ids, "No users have positive validation ratings."

liked_training_users = set(train_ratings.loc[
    train_ratings["rating"] >= POSITIVE_THRESHOLD, "userId"
])
training_movie_ids = set(train_ratings["movieId"])

print("Validation users:", validation_ratings["userId"].nunique())
print("Users with positive validation ratings:", len(evaluation_user_ids))
print("Users excluded because they have no validation positives:",
      validation_ratings["userId"].nunique() - len(evaluation_user_ids))
print("Eligible users with liked training history:",
      sum(user_id in liked_training_users for user_id in evaluation_user_ids))
print("Positive validation ratings for movies absent from training:",
      (~validation_positives["movieId"].isin(training_movie_ids)).sum())


## Fit the popularity baseline

Count rating events in training, irrespective of star rating. This baseline measures how frequently a movie was rated and learns no personalized preference weights. Rank by descending training count, then ascending `movieId` for deterministic ties. Validation and test interactions never contribute to these counts.


In [ ]:
movie_popularity = (
    train_ratings.groupby("movieId")
    .size()
    .reset_index(name="training_rating_count")
    .merge(movies[["movieId", "title"]], on="movieId", validate="one_to_one")
    .sort_values(["training_rating_count", "movieId"], ascending=[False, True])
    .reset_index(drop=True)
)
assert movie_popularity["training_rating_count"].sum() == len(train_ratings)
assert set(movie_popularity["movieId"]) == training_movie_ids
display(movie_popularity.head(TOP_K))


## Generate recommendations from observed history

The baseline shares one popularity ranking across users, but its eligible list differs by user: exclude **every** movie they rated during fitting, including disliked movies. The same rule is reused by the personalized models. The function returns up to `n` recommendations when fewer unseen candidates are available.


In [ ]:
def recommend_popular(user_id, training_data, popularity_ranking, n=10):
    """Return the highest-ranked fitting-catalog movies not rated by this user."""
    seen_ids = training_data.loc[
        training_data["userId"] == user_id, "movieId"
    ]
    return (
        popularity_ranking.loc[~popularity_ranking["movieId"].isin(seen_ids)]
        .head(n)
        .reset_index(drop=True)
    )


## Inspect one evaluable existing user

Select the first validation-positive user deterministically. The previous global-split demonstration selected a user without training history; that case no longer exists in this benchmark. Display the recommendations alongside the user's held-out positives to make the recall calculation inspectable. Held-out labels are used only to score the recommendations, never to construct them.


In [ ]:
example_user_id = evaluation_user_ids[0]
example_recommendations = recommend_popular(
    example_user_id, train_ratings, movie_popularity, n=TOP_K
)
seen_ids = set(train_ratings.loc[
    train_ratings["userId"] == example_user_id, "movieId"
])
positive_ids = set(validation_positives.loc[
    validation_positives["userId"] == example_user_id, "movieId"
])
assert set(example_recommendations["movieId"]).isdisjoint(seen_ids)

example_recommendations["positive_in_validation"] = (
    example_recommendations["movieId"].isin(positive_ids)
)
print("Example user:", example_user_id)
print("Training movies excluded:", len(seen_ids))
print("Positive validation movies:", len(positive_ids))
print("Positive validation movies in the training catalog:",
      len(positive_ids.intersection(training_movie_ids)))
print("Recall@10:",
      example_recommendations["positive_in_validation"].sum() / len(positive_ids))
display(example_recommendations)


## Evaluate Recall@10 across the validation cohort

For each eligible user, count how many of their positively rated validation movies occur in the top ten recommendations and divide by the number of their validation positives. Average these per-user recalls with equal user weight (macro averaging). Users without positive validation ratings are excluded because their recall denominator is zero.

The evaluator also checks candidate membership, uniqueness, and seen-movie exclusions. Keeping unavailable positives in the denominator makes candidate coverage a visible limitation shared by all models.


In [ ]:
evaluation_rows = []
training_users = set(train_ratings["userId"])

for user_id, positives in validation_positives.groupby("userId", sort=True):
    positive_ids = set(positives["movieId"])
    recommendations = recommend_popular(
        user_id, train_ratings, movie_popularity, n=TOP_K
    )
    recommended_ids = set(recommendations["movieId"])
    seen_ids = set(train_ratings.loc[
        train_ratings["userId"] == user_id, "movieId"
    ])
    assert recommendations["movieId"].is_unique
    assert len(recommendations) <= TOP_K
    assert recommended_ids <= training_movie_ids
    assert recommended_ids.isdisjoint(seen_ids)

    hits = len(positive_ids.intersection(recommended_ids))
    evaluation_rows.append({
        "userId": user_id,
        "positive_movie_count": len(positive_ids),
        "hits_at_10": hits,
        "recall_at_10": hits / len(positive_ids),
        "has_training_history": user_id in training_users,
    })

popularity_results = pd.DataFrame(evaluation_rows)
assert popularity_results["has_training_history"].all()
assert popularity_results["userId"].is_unique
print("Users evaluated:", len(popularity_results))
print("Mean validation Recall@10:", popularity_results["recall_at_10"].mean())
display(popularity_results.head())


## Save the validation benchmark

Persist per-user results with the split fingerprint, positive threshold, and recommendation count. Later notebooks verify this provenance before comparing models. No numerical conclusion from the former global split is carried forward: the generated table is the baseline for this protocol.


In [ ]:
save_validation_results(
    popularity_results, "popularity_validation_results.csv"
)
print("Saved popularity validation results for:", SPLIT_ID)


## Interpretation and review notes

The popularity baseline supplies the reference for evaluating personalization under the same candidate and metric rules. Its Recall@10 measures recovery of recorded positive feedback, not the quality of every recommended but unrated movie. Unobserved preferences remain unknown, and users with many held-out positives face a smaller attainable recall under a ten-item recommendation limit.

Notebook 02 fits item-based collaborative filtering using these saved training interactions. Notebooks 02–05 compare alternatives on validation only; model settings must be fixed before fitting on training plus validation and assessing the reserved test interactions. Re-run downstream notebooks whenever the source data, split implementation, or modeling settings change.
